# IndoBERT Token Classifier (Colab)

Notebook ini untuk inferensi label token pantun modern dengan skema label BIO (`B/I-ASG`, `B/I-SKT`, `B/I-DSM`, `B/I-TYP`, `O`) memakai model IndoBERT yang disimpan di Google Drive.

Sesuai dokumentasi pipeline:
- `B/I-SKT` -> Rumus A (fonemisasi alfabetis)
- `B/I-DSM` / `B/I-TYP` -> Rumus B & C
- `B/I-ASG` -> failsafe G2P asing
- `O` -> Cascaded CRF 1 (G2P kata baku)


In [1]:
!pip -q install transformers sentencepiece accelerate g2p-id g2p-en nltk symspellpy sklearn-crfsuite joblib

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# UBAH sesuai lokasi folder model fine-tuned Anda di Google Drive.
# Folder ini minimal berisi: config.json, pytorch_model.bin/model.safetensors,
# tokenizer.json atau vocab.txt + tokenizer config.
MODEL_DIR = '/content/drive/MyDrive/Models/indobert-pantun-ner'  # <- ganti path ini

# Kamus kata dasar (1 kata per baris) untuk SymSpell recover typo B/I-TYP.
# Silakan taruh file ini di Google Drive Anda.
KATA_DASAR_PATH = '/content/drive/MyDrive/kata-dasar.txt'  # <- ganti bila perlu

# Model CRF layer-3 (syllabifier B/I dari deret fonem hasil G2P).
CRF_BI_MODEL_PATH = '/content/drive/MyDrive/Models/crf_layer3_bi.joblib'  # <- ganti bila perlu


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()

id2label = model.config.id2label
label2id = model.config.label2id
print('Labels:', id2label)

In [ ]:
def predict_token_labels(text: str):
    """Inferensi token-level label + confidence dengan penyelarasan wordpiece -> token asli."""
    words = text.strip().split()
    enc = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors='pt',
        truncation=True,
        padding=False
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1)[0]
        pred_ids = torch.argmax(logits, dim=-1)[0].tolist()

    word_ids = tokenizer(words, is_split_into_words=True).word_ids()

    token_rows = []
    current_word = None
    for idx, widx in enumerate(word_ids):
        if widx is None:
            continue
        if widx != current_word:
            current_word = widx
            label = id2label[pred_ids[idx]]

            top2 = torch.topk(probs[idx], k=min(2, probs[idx].shape[-1]))
            confidence = float(top2.values[0].item())
            second_best = float(top2.values[1].item()) if top2.values.shape[0] > 1 else 0.0
            margin = confidence - second_best

            token_rows.append(
                {
                    'token': words[widx],
                    'label': label,
                    'confidence': confidence,
                    'margin': margin,
                }
            )
    return token_rows


LABELS_ASING = {"B-ASG", "I-ASG"}
LABELS_SINGKATAN = {"B-SKT", "I-SKT"}
LABELS_DISEMVOWEL = {"B-DSM", "I-DSM"}
LABELS_TYPO = {"B-TYP", "I-TYP"}
LABELS_VALID = {"O", *LABELS_ASING, *LABELS_SINGKATAN, *LABELS_DISEMVOWEL, *LABELS_TYPO}


def route_by_label(label: str) -> str:
    """Routing sesuai dokumentasi pipeline dan skema label BIO IndoBERT."""
    norm = label.upper().strip()
    if norm not in LABELS_VALID:
        raise ValueError(f'Label tidak dikenali: {label}')

    if norm in LABELS_SINGKATAN:
        return 'RUMUS_A_FONEMISASI_ALFABETIS'
    if norm in LABELS_DISEMVOWEL or norm in LABELS_TYPO:
        return 'RUMUS_B_C_FONEMISASI_DISEMVOWELLING'
    if norm in LABELS_ASING:
        return 'FAILSAFE_G2P_ASING'
    return 'CRF1_G2P_KATA_BAKU'


def is_likely_disemvowelling(token: str) -> bool:
    """Heuristik robust untuk mendeteksi pola disemvowelling tanpa kamus hardcoded."""
    t = token.lower().strip()
    if not t.isalpha() or len(t) < 3:
        return False

    vowels = set('aiueo')
    vowel_count = sum(ch in vowels for ch in t)
    consonant_ratio = (len(t) - vowel_count) / len(t)

    max_cons_run = 0
    curr_run = 0
    for ch in t:
        if ch in vowels:
            curr_run = 0
        else:
            curr_run += 1
            max_cons_run = max(max_cons_run, curr_run)

    consonant_clusters = (
        'ng', 'ny', 'kh', 'sy', 'tr', 'br', 'gr', 'pr', 'kr', 'sk', 'st', 'nd', 'mb'
    )
    cluster_hits = sum(1 for c in consonant_clusters if c in t)

    score = 0
    if vowel_count == 0:
        score += 3
    elif vowel_count == 1 and len(t) >= 4:
        score += 2
    if consonant_ratio >= 0.70:
        score += 2
    if max_cons_run >= 3:
        score += 2
    if cluster_hits >= 1:
        score += 1
    if len(t) >= 4 and t[0] not in vowels and t[-1] not in vowels and vowel_count <= 1:
        score += 1

    return score >= 4


def override_typo_to_dsm(token: str, label: str) -> str:
    norm = label.upper().strip()
    if norm in LABELS_TYPO and is_likely_disemvowelling(token):
        recover_fn = globals().get('symspell_recover_typo')
        if callable(recover_fn):
            recovered = recover_fn(token)
            if recovered:
                return norm
        return norm.replace('TYP', 'DSM')
    return norm


def predict_with_routing(text: str):
    labeled = predict_token_labels(text)
    rows = []
    for item in labeled:
        tok = item['token']
        lab = item['label']
        token_meta = {
            'confidence': item.get('confidence'),
            'margin': item.get('margin'),
        }

        final_label = override_typo_to_dsm(tok, lab)
        switch = acoustic_switchboard(tok, final_label, token_meta=token_meta)
        bi_fn = globals().get('predict_bi_from_phoneme')
        bi_pattern = bi_fn(switch['phoneme']) if callable(bi_fn) else None

        rows.append(
            {
                'token': tok,
                'raw_label': lab,
                'label': final_label,
                'route': route_by_label(final_label),
                'model_confidence': token_meta['confidence'],
                'model_margin': token_meta['margin'],
                'switch_route': switch['switch_route'],
                'switch_input': switch['switch_input'],
                'phoneme': switch['phoneme'],
                'bi_pattern': bi_pattern,
                'bi_source': 'CRF3_BI' if bi_pattern else None,
                'bc_info': switch.get('bc_info'),
                'asg_info': switch.get('asg_info')
            }
        )
    return rows


In [ ]:
# Setup/training SymSpell dari kamus kata dasar (jalankan sebelum cell helper)
import os
from symspellpy import SymSpell


def _resolve_kata_dasar_path() -> str:
    configured = globals().get('KATA_DASAR_PATH', '/content/drive/MyDrive/kata-dasar.txt')
    return str(configured).strip()


def build_symspell_from_kata_dasar(dict_path: str, max_edit_distance: int = 2):
    if not dict_path or not os.path.exists(dict_path):
        print(f'[SymSpell] Dictionary tidak ditemukan: {dict_path}. Recover typo B/I-TYP dinonaktifkan.')
        return None

    symspell = SymSpell(max_dictionary_edit_distance=max_edit_distance, prefix_length=7)
    loaded = 0

    with open(dict_path, 'r', encoding='utf-8') as f:
        for raw in f:
            term = raw.strip().lower()
            if not term or not term.isalpha():
                continue
            symspell.create_dictionary_entry(term, 1)
            loaded += 1

    if loaded == 0:
        print(f'[SymSpell] File kamus kosong/tidak valid: {dict_path}. Recover typo B/I-TYP dinonaktifkan.')
        return None

    print(f'[SymSpell] Loaded {loaded} kata dari: {dict_path}')
    return symspell


symspell_model = build_symspell_from_kata_dasar(_resolve_kata_dasar_path())


In [ ]:
# Helper sederhana Rumus A (singkatan)
from functools import lru_cache
import os
import re
import nltk
import joblib
from symspellpy import Verbosity
from g2p_id import G2P as G2P_ID
from g2p_en import G2p as G2P_EN


def _ensure_nltk_resource(resource_path: str, package_name: str):
    try:
        nltk.data.find(resource_path)
    except LookupError:
        nltk.download(package_name, quiet=True)


_ensure_nltk_resource('corpora/cmudict', 'cmudict')
_ensure_nltk_resource('taggers/averaged_perceptron_tagger_eng', 'averaged_perceptron_tagger_eng')
_ensure_nltk_resource('taggers/averaged_perceptron_tagger', 'averaged_perceptron_tagger')

g2p_id_model = G2P_ID()
g2p_en_model = G2P_EN()

ALPHABET_EJA_MAP = {
    "a": "a", "b": "be", "c": "ce", "d": "de", "e": "e", "f": "ef", "g": "ge",
    "h": "ha", "i": "i", "j": "je", "k": "ka", "l": "el", "m": "em", "n": "en",
    "o": "o", "p": "pe", "q": "ki", "r": "er", "s": "es", "t": "te", "u": "u",
    "v": "ve", "w": "we", "x": "eks", "y": "ye", "z": "zet"
}

# Konversi simbol fonem Inggris (ARPAbet) ke fonetik internasional (IPA)
ARPABET_TO_IPA = {
    "AA": "ɑ", "AE": "æ", "AH": "ʌ", "AO": "ɔ", "AW": "aʊ", "AY": "aɪ",
    "B": "b", "CH": "tʃ", "D": "d", "DH": "ð", "EH": "ɛ", "ER": "ɝ",
    "EY": "eɪ", "F": "f", "G": "ɡ", "HH": "h", "IH": "ɪ", "IY": "i",
    "JH": "dʒ", "K": "k", "L": "l", "M": "m", "NG": "ŋ", "OW": "oʊ",
    "OY": "ɔɪ", "P": "p", "R": "r", "S": "s", "SH": "ʃ", "T": "t",
    "TH": "θ", "UH": "ʊ", "UW": "u", "V": "v", "W": "w", "Y": "j",
    "Z": "z", "ZH": "ʒ"
}




CRF3_VOWELS = {"a", "i", "u", "e", "o", "ə", "ɛ", "ɪ", "ʊ", "ɔ", "æ", "ɐ", "ɒ"}
crf_bi_model = None


def load_crf_bi_model(model_path=None):
    """Load model CRF layer-3 (.joblib)."""
    global crf_bi_model
    target = str(model_path or globals().get('CRF_BI_MODEL_PATH', '')).strip()
    if not target:
        print('[CRF3] Path model kosong. Prediksi BI dinonaktifkan.')
        crf_bi_model = None
        return None
    if not os.path.exists(target):
        print(f'[CRF3] Model tidak ditemukan: {target}. Prediksi BI dinonaktifkan.')
        crf_bi_model = None
        return None

    crf_bi_model = joblib.load(target)
    print(f'[CRF3] Loaded model: {target}')
    return crf_bi_model


CRF3_DIPHTHONGS = ("eɪ", "aɪ", "aʊ", "ɔɪ", "oʊ")


def phoneme_to_units(phoneme: str) -> list[str]:
    text = str(phoneme).strip()
    if not text:
        return []
    if ' ' in text:
        return [u for u in text.split() if u]

    chars = [ch for ch in text if not ch.isspace()]
    units: list[str] = []
    i = 0
    while i < len(chars):
        if i + 1 < len(chars):
            pair = chars[i] + chars[i + 1]
            if pair in CRF3_DIPHTHONGS:
                units.append(pair)
                i += 2
                continue
        units.append(chars[i])
        i += 1
    return units


def _crf3_is_vowel(phone: str) -> bool:
    return phone in CRF3_VOWELS


def _crf3_phone_features(seq: list[str], i: int) -> dict:
    cur = seq[i]
    prev = seq[i - 1] if i > 0 else '<BOS>'
    nxt = seq[i + 1] if i < len(seq) - 1 else '<EOS>'
    return {
        'bias': 1.0,
        'phone': cur,
        'prev_phone': prev,
        'next_phone': nxt,
        'bigram_prev_cur': f'{prev}+{cur}',
        'bigram_cur_next': f'{cur}+{nxt}',
        'is_vowel': _crf3_is_vowel(cur),
        'prev_is_vowel': False if prev == '<BOS>' else _crf3_is_vowel(prev),
        'next_is_vowel': False if nxt == '<EOS>' else _crf3_is_vowel(nxt),
        'pos': i,
        'rev_pos': len(seq) - 1 - i,
        'len_seq': len(seq),
        'bos': i == 0,
        'eos': i == len(seq) - 1,
    }


def predict_bi_from_phoneme(phoneme: str, model=None):
    mdl = model if model is not None else globals().get('crf_bi_model')
    if mdl is None:
        return None

    units = phoneme_to_units(phoneme)
    if not units:
        return None

    feats = [_crf3_phone_features(units, i) for i in range(len(units))]
    pred = mdl.predict_single(feats)
    return ''.join(pred)

ASG_EN_PRIORITY = {
    'meeting', 'download', 'online', 'glowing', 'makeup', 'chat', 'launching', 'booster'
}


def _count_syllables_from_bi(bi_pattern: str | None) -> int:
    return sum(1 for ch in (bi_pattern or '') if ch == 'B')


def hitung_skor_kewajaran(units: list[str], label_route: str, bi_pattern: str | None = None) -> float:
    """Skor kelayakan fonotaktik untuk konteks pelafalan Indonesia; lebih kecil lebih wajar."""
    if not units:
        return 999.0

    skor = 0.0
    diftong_asing = set(CRF3_DIPHTHONGS)
    jumlah_diftong = sum(1 for u in units if u in diftong_asing)
    if jumlah_diftong > 1:
        skor += 3.0

    for i in range(len(units) - 2):
        is_c0 = not _crf3_is_vowel(units[i])
        is_c1 = not _crf3_is_vowel(units[i + 1])
        is_c2 = not _crf3_is_vowel(units[i + 2])
        if is_c0 and is_c1 and is_c2:
            skor += 2.0

    syll_count = _count_syllables_from_bi(bi_pattern)
    if syll_count <= 1 and len(units) >= 4:
        skor += 2.5

    vowel_count = sum(1 for u in units if _crf3_is_vowel(u))
    if vowel_count == 0:
        skor += 2.0

    if label_route == 'EN' and syll_count >= 5:
        skor += 1.0

    return float(skor)


def jalur_asg_hybrid(token: str, token_meta: dict | None = None) -> dict:
    """Dual-candidate routing untuk ASG: bandingkan kandidat G2P-EN dan G2P-ID."""
    token_clean = normalize_typo_token(token)
    if not token_clean:
        return {
            'switch_route': 'ASG_G2P_EN_IPA',
            'switch_input': token_clean,
            'phoneme': '',
            'asg_info': {'reason': 'empty_token'},
        }

    asg_en = g2p_foreign_phoneme(token_clean)
    asg_id = g2p_id_phoneme(token_clean)

    units_en = phoneme_to_units(asg_en)
    units_id = phoneme_to_units(asg_id)

    bi_en = predict_bi_from_phoneme(asg_en)
    bi_id = predict_bi_from_phoneme(asg_id)

    skor_en = hitung_skor_kewajaran(units_en, 'EN', bi_en)
    skor_id = hitung_skor_kewajaran(units_id, 'ID', bi_id)

    syll_en = _count_syllables_from_bi(bi_en)
    syll_id = _count_syllables_from_bi(bi_id)
    confidence = None if not token_meta else token_meta.get('confidence')

    # Soft priority: istilah modern tetap prefer EN, kecuali EN kolaps jadi 1 silabel pada token panjang.
    if token_clean in ASG_EN_PRIORITY:
        en_collapsed = syll_en <= 1 and len(units_en) >= 4
        id_healthier = syll_id >= 2 and len(units_id) >= 3
        if en_collapsed and id_healthier:
            return {
                'switch_route': 'ASG_G2P_ID_FALLBACK',
                'switch_input': token_clean,
                'phoneme': asg_id,
                'asg_info': {
                    'reason': 'en_priority_overridden_by_syllable_guard',
                    'token_clean': token_clean,
                    'score_en': skor_en,
                    'score_id': skor_id,
                    'bi_en': bi_en,
                    'bi_id': bi_id,
                    'syll_en': syll_en,
                    'syll_id': syll_id,
                    'units_en': units_en,
                    'units_id': units_id,
                },
            }

        return {
            'switch_route': 'ASG_G2P_EN_IPA',
            'switch_input': token_clean,
            'phoneme': asg_en,
            'asg_info': {
                'reason': 'en_priority',
                'token_clean': token_clean,
                'score_en': skor_en,
                'score_id': skor_id,
                'bi_en': bi_en,
                'bi_id': bi_id,
                'syll_en': syll_en,
                'syll_id': syll_id,
                'units_en': units_en,
                'units_id': units_id,
            },
        }

    if abs(skor_en - skor_id) <= 0.25:
        prefer_en = confidence is not None and float(confidence) >= 0.98
    else:
        prefer_en = skor_en < skor_id

    if prefer_en:
        return {
            'switch_route': 'ASG_G2P_EN_IPA',
            'switch_input': token_clean,
            'phoneme': asg_en,
            'asg_info': {
                'reason': 'score_en_better',
                'token_clean': token_clean,
                'score_en': skor_en,
                'score_id': skor_id,
                'bi_en': bi_en,
                'bi_id': bi_id,
                'syll_en': syll_en,
                'syll_id': syll_id,
                'units_en': units_en,
                'units_id': units_id,
            },
        }

    return {
        'switch_route': 'ASG_G2P_ID_FALLBACK',
        'switch_input': token_clean,
        'phoneme': asg_id,
        'asg_info': {
            'reason': 'score_id_better',
            'token_clean': token_clean,
            'score_en': skor_en,
            'score_id': skor_id,
            'bi_en': bi_en,
            'bi_id': bi_id,
            'syll_en': syll_en,
            'syll_id': syll_id,
            'units_en': units_en,
            'units_id': units_id,
        },
    }


def singkatan_to_ejaan(word: str) -> str:
    """dpr -> de pe er"""
    clean = "".join(ch for ch in word.lower() if ch.isalpha())
    return " ".join(ALPHABET_EJA_MAP.get(ch, ch) for ch in clean)


def arpabet_to_ipa(arpabet_tokens: list[str]) -> str:
    ipa_tokens = []
    for token in arpabet_tokens:
        base = re.sub(r"\d", "", token).upper()
        if not base:
            continue
        ipa_tokens.append(ARPABET_TO_IPA.get(base, base.lower()))
    return "".join(ipa_tokens)


def _coerce_g2p_output_to_str(raw) -> str:
    if isinstance(raw, tuple):
        return _coerce_g2p_output_to_str(raw[0])
    if isinstance(raw, list):
        if raw and isinstance(raw[0], list):
            raw = [ph for sub in raw for ph in sub if ph]
        return " ".join(str(part) for part in raw if part)
    return str(raw).strip()


@lru_cache(maxsize=8192)
def g2p_id_phoneme(word: str) -> str:
    if not word:
        return ""
    try:
        res = g2p_id_model.to_phoneme(word)
    except AttributeError:
        try:
            res = g2p_id_model(word)
        except Exception:
            res = g2p_id_model.convert(word)
    return _coerce_g2p_output_to_str(res).replace(" ", "")


@lru_cache(maxsize=8192)
def g2p_foreign_phoneme(word: str) -> str:
    if not word:
        return ""
    raw = g2p_en_model(word)
    phones = [p for p in raw if p and p != " " and p != "<unk>"]
    ipa = arpabet_to_ipa(phones)
    return ipa if ipa else g2p_id_phoneme(word)


def rumus_a_singkatan_simple(word: str) -> str:
    """Rumus A: singkatan -> ejaan alfabet -> fonem Indonesia (g2p-id)."""
    ejaan = singkatan_to_ejaan(word)
    if not ejaan:
        return ""
    return g2p_id_phoneme(ejaan)


# -------------------------------
# Rumus B & C + typo handling (tanpa hardcoded kata)
# -------------------------------

VOWELS = set('aiueo')
COMMON_DIGRAPHS = ("ng", "ny", "kh", "sy", "br", "tr", "kr", "pr", "gr", "dr", "st", "sk")


def normalize_typo_token(token: str) -> str:
    """Normalisasi typo berbasis bentuk: lowercase, alfabet-only, kompres pengulangan huruf."""
    t = token.lower().strip()
    if not t:
        return t
    t = "".join(ch for ch in t if ch.isalpha())
    if not t:
        return t
    t = re.sub(r"(.)\1{2,}", r"\1\1", t)
    return t


@lru_cache(maxsize=8192)
def symspell_recover_typo(
    token: str,
    allow_no_vowel: bool = False,
    max_edit_distance: int = 2,
    max_len_delta: int = 3,
) -> dict | None:
    """Recover typo ringan; return None jika tidak yakin."""
    model = globals().get('symspell_model')
    if model is None:
        return None

    normalized = normalize_typo_token(token)
    if not normalized:
        return None

    if (not allow_no_vowel) and all(ch not in VOWELS for ch in normalized):
        return None

    suggestions = model.lookup(
        normalized,
        Verbosity.TOP,
        max_edit_distance=max_edit_distance,
        include_unknown=True,
        transfer_casing=False,
    )
    if not suggestions:
        return None

    top = suggestions[0]
    recovered = top.term.lower().strip()
    if not recovered or recovered == normalized:
        return None

    allowed_dist = 1 if len(normalized) <= 5 else 2
    allowed_dist = min(allowed_dist, max_edit_distance)
    if top.distance > allowed_dist:
        return None
    if abs(len(recovered) - len(normalized)) > max_len_delta:
        return None

    return {
        'original': normalized,
        'recovered': recovered,
        'distance': int(top.distance),
        'count': int(top.count),
    }


def _max_consonant_run(t: str) -> int:
    best = 0
    run = 0
    for ch in t:
        if ch in VOWELS:
            run = 0
        else:
            run += 1
            best = max(best, run)
    return best


def _has_common_cluster(t: str) -> bool:
    return any(dg in t for dg in COMMON_DIGRAPHS)


def _has_reversed_common_cluster(t: str) -> bool:
    reversed_digraphs = {dg[::-1] for dg in COMMON_DIGRAPHS if len(dg) == 2}
    return any(rdg in t for rdg in reversed_digraphs)


def _is_content_like_disemvowelling(t: str) -> bool:
    vowel_count = sum(ch in VOWELS for ch in t)
    cons_run = _max_consonant_run(t)
    if vowel_count == 0:
        return True
    if vowel_count <= 1 and len(t) >= 4:
        return True
    if cons_run >= 3:
        return True
    if cons_run >= 2 and _has_common_cluster(t):
        return True
    return False


def is_low_confidence_label(token_meta: dict | None, conf_threshold: float = 0.80, margin_threshold: float = 0.45) -> bool:
    if not token_meta:
        return False
    confidence = token_meta.get('confidence')
    margin = token_meta.get('margin')
    if confidence is None or margin is None:
        return False
    return float(confidence) <= conf_threshold or float(margin) <= margin_threshold


def should_attempt_o_recover(token: str, token_meta: dict | None = None) -> dict:
    normalized = normalize_typo_token(token)
    if len(normalized) < 3:
        return {'suspicious': False, 'normalized': normalized}

    cons_run = _max_consonant_run(normalized)
    has_common_cluster = _has_common_cluster(normalized)
    has_reversed_cluster = _has_reversed_common_cluster(normalized)
    low_conf = is_low_confidence_label(token_meta)

    suspicious = cons_run >= 2 and (not has_common_cluster) and (low_conf or has_reversed_cluster)
    return {
        'suspicious': suspicious,
        'normalized': normalized,
        'cons_run': cons_run,
        'has_common_cluster': has_common_cluster,
        'has_reversed_cluster': has_reversed_cluster,
        'low_confidence': low_conf,
    }


def should_attempt_typo_recover(token: str, token_meta: dict | None = None) -> dict:
    normalized = normalize_typo_token(token)
    if len(normalized) < 3:
        return {
            'attempt_recover': False,
            'use_bc': False,
            'normalized': normalized,
        }

    vowel_count = sum(ch in VOWELS for ch in normalized)
    cons_run = _max_consonant_run(normalized)
    has_reversed_cluster = _has_reversed_common_cluster(normalized)
    disemvowel_like = _is_content_like_disemvowelling(normalized)
    sparse_vowel = len(normalized) >= 4 and vowel_count <= 1
    low_conf = is_low_confidence_label(token_meta)

    # Untuk label TYP, BC hanya dipakai bila benar-benar ada sinyal typo/disemvowelling kuat.
    # Ini mencegah kata normal panjang (mis. berpacarpacaran/berenangrenang) jatuh ke template vonem*.
    typo_strong_signal = sparse_vowel or has_reversed_cluster or low_conf
    attempt_recover = disemvowel_like and typo_strong_signal
    use_bc = disemvowel_like and typo_strong_signal

    return {
        'attempt_recover': attempt_recover,
        'use_bc': use_bc,
        'normalized': normalized,
        'vowel_count': vowel_count,
        'cons_run': cons_run,
        'has_reversed_cluster': has_reversed_cluster,
        'disemvowel_like': disemvowel_like,
        'sparse_vowel': sparse_vowel,
        'low_confidence': low_conf,
    }


def rumus_c_clamping(word: str, force_two_syllable: bool = False) -> str:
    """Rumus C: clamping panjang kata; opsional dipaksa 2 unit untuk typo pendek."""
    t = word.lower().strip()
    if force_two_syllable:
        return "vone" if t and t[-1] in VOWELS else "vonem"
    if len(t) <= 2:
        return "von"
    if len(t) == 3:
        return "vone" if t and t[-1] in VOWELS else "vonem"
    return "vone" if t and t[-1] in VOWELS else "vonem"


def rumus_b_phonotactic_mapping(word: str) -> str:
    """Rumus B: estimasi struktur fonotaktik untuk kata konten tanpa vokal."""
    t = "".join(ch for ch in word.lower() if ch.isalpha())
    if not t:
        return "von"

    vowel_count = sum(ch in VOWELS for ch in t)
    cons_run = _max_consonant_run(t)
    has_cluster = _has_common_cluster(t)

    if vowel_count == 0:
        sonorants = set('rlmnwy')
        if len(t) == 3 and any(ch in sonorants for ch in t):
            return "vonemi"
        if len(t) <= 4:
            syll_est = 2
        elif has_cluster or cons_run >= 4:
            syll_est = 3
        else:
            syll_est = 2
        patterns = {2: "vonem", 3: "vonemis", 4: "vonemisa"}
        return patterns[syll_est]

    compact = t
    for dg in COMMON_DIGRAPHS:
        compact = compact.replace(dg, "X")

    base_units = len(compact)
    has_coda = t[-1] in "bcdfghjklmnpqrstvwxyz"
    if has_coda and base_units > 1:
        base_units -= 1

    syll_est = max(2, min(4, (base_units + 1) // 2))

    patterns = {
        2: "vonem",
        3: "vonemis",
        4: "vonemisa",
    }
    return patterns[syll_est]


def rumus_bc_for_dsm_typo(token: str, label: str) -> dict:
    """
    Bedanya:
    - Rumus C: clamping sederhana (panjang token) untuk kata tugas/pendek.
    - Rumus B: mapping fonotaktik untuk kata konten/disemvowelling kompleks.
    """
    norm_label = label.upper().strip()
    normalized_token = normalize_typo_token(token)

    # Kasus pronoun gaul: gw/gue/gua dipetakan ke 2 ketuk (vone).
    if normalized_token in {"gw", "gue", "gua"}:
        spoken = "gue" if normalized_token == "gw" else normalized_token
        return {
            "normalized_token": spoken,
            "rule_used": "C",
            "phoneme": rumus_c_clamping(spoken, force_two_syllable=True),
        }

    if norm_label in LABELS_TYPO and len(normalized_token) <= 3:
        return {
            "normalized_token": normalized_token,
            "rule_used": "C",
            "phoneme": rumus_c_clamping(normalized_token, force_two_syllable=True),
        }

    if len(normalized_token) <= 2 and all(ch not in VOWELS for ch in normalized_token):
        return {
            "normalized_token": normalized_token,
            "rule_used": "C",
            "phoneme": rumus_c_clamping(normalized_token),
        }

    if _is_content_like_disemvowelling(normalized_token):
        return {
            "normalized_token": normalized_token,
            "rule_used": "B",
            "phoneme": rumus_b_phonotactic_mapping(normalized_token),
        }

    if len(normalized_token) <= 3:
        return {
            "normalized_token": normalized_token,
            "rule_used": "C",
            "phoneme": rumus_c_clamping(normalized_token),
        }

    return {
        "normalized_token": normalized_token,
        "rule_used": "C",
        "phoneme": rumus_c_clamping(normalized_token),
    }


def acoustic_switchboard(token: str, label: str, token_meta: dict | None = None) -> dict:
    """Switch board fonem: jalur A, B/C, ASG, atau O sesuai label IndoBERT."""
    norm = label.upper().strip()
    cleaned = normalize_typo_token(token)

    if norm in LABELS_SINGKATAN:
        ejaan = singkatan_to_ejaan(token)
        return {
            "switch_route": "A_SINGKATAN_G2P_ID",
            "switch_input": ejaan,
            "phoneme": rumus_a_singkatan_simple(token),
        }

    if norm in LABELS_ASING:
        return jalur_asg_hybrid(token, token_meta=token_meta)

    if norm in LABELS_TYPO:
        typ_gate = should_attempt_typo_recover(token, token_meta=token_meta)

        if typ_gate['attempt_recover']:
            typo_edit_distance = 2 if typ_gate.get('has_reversed_cluster') else 1
            recovered = symspell_recover_typo(
                token,
                allow_no_vowel=False,
                max_edit_distance=typo_edit_distance,
                max_len_delta=2,
            )
            if recovered:
                return {
                    "switch_route": "TYP_SYMSPELL_G2P_ID",
                    "switch_input": recovered['recovered'],
                    "phoneme": g2p_id_phoneme(recovered['recovered']),
                    "bc_info": {
                        "rule_used": "SYM",
                        "normalized_token": recovered['recovered'],
                        "recover": recovered,
                        "gate": typ_gate,
                    },
                }

        if typ_gate['use_bc']:
            bc = rumus_bc_for_dsm_typo(token, norm)
            return {
                "switch_route": f"BC_{bc['rule_used']}",
                "switch_input": bc["normalized_token"],
                "phoneme": bc["phoneme"],
                "bc_info": bc,
            }

        return {
            "switch_route": "TYP_G2P_ID_DIRECT",
            "switch_input": cleaned,
            "phoneme": g2p_id_phoneme(cleaned),
            "bc_info": {
                "rule_used": "DIRECT",
                "normalized_token": cleaned,
                "gate": typ_gate,
            },
        }

    if norm in LABELS_DISEMVOWEL:
        bc = rumus_bc_for_dsm_typo(token, norm)
        return {
            "switch_route": f"BC_{bc['rule_used']}",
            "switch_input": bc["normalized_token"],
            "phoneme": bc["phoneme"],
            "bc_info": bc,
        }

    gate = should_attempt_o_recover(token, token_meta=token_meta)
    if gate['suspicious']:
        recovered_o = symspell_recover_typo(
            token,
            allow_no_vowel=True,
            max_edit_distance=2,
            max_len_delta=2,
        )
        is_strict_confident = recovered_o and recovered_o['distance'] <= 1 and recovered_o['count'] >= 2
        is_transposition_confident = recovered_o and gate.get('has_reversed_cluster') and recovered_o['distance'] <= 2 and recovered_o['count'] >= 1
        if is_strict_confident or is_transposition_confident:
            return {
                "switch_route": "O_SYMSPELL_G2P_ID",
                "switch_input": recovered_o['recovered'],
                "phoneme": g2p_id_phoneme(recovered_o['recovered']),
                "bc_info": {
                    "rule_used": "O_SYM",
                    "normalized_token": recovered_o['recovered'],
                    "recover": recovered_o,
                    "gate": gate,
                },
            }

    return {
        "switch_route": "O_G2P_ID",
        "switch_input": cleaned,
        "phoneme": g2p_id_phoneme(cleaned),
    }

load_crf_bi_model(globals().get('CRF_BI_MODEL_PATH'))

def _decode_bi_to_syllables(units: list[str], bi_pattern: str | None) -> dict:
    if not units:
        return {
            'valid': False,
            'labels': [],
            'syllables_units': [],
            'syllables_text': [],
            'error': 'units_empty',
        }

    labels = list((bi_pattern or '').strip())
    if len(labels) != len(units) or any(lb not in {'B', 'I'} for lb in labels):
        syll_text = ''.join(units)
        return {
            'valid': False,
            'labels': labels,
            'syllables_units': [units],
            'syllables_text': [syll_text],
            'error': 'invalid_bi_pattern',
        }

    syllables_units: list[list[str]] = []
    current: list[str] = []
    for i, (u, lb) in enumerate(zip(units, labels)):
        if i == 0 or lb == 'B':
            if current:
                syllables_units.append(current)
            current = [u]
        else:
            current.append(u)
    if current:
        syllables_units.append(current)

    syllables_text = [''.join(parts) for parts in syllables_units]
    return {
        'valid': True,
        'labels': labels,
        'syllables_units': syllables_units,
        'syllables_text': syllables_text,
        'error': None,
    }


def _extract_rhyme_tail(syllable_text: str) -> str:
    if not syllable_text:
        return ''
    chars = [ch for ch in syllable_text if not ch.isspace()]
    if not chars:
        return ''
    for i in range(len(chars) - 1, -1, -1):
        if chars[i] in CRF3_VOWELS:
            return ''.join(chars[i:])
    return ''.join(chars)


def analyze_line_prosody_full(text: str) -> dict:
    token_rows = predict_with_routing(text)
    analyzed_tokens = []
    line_syllables = []
    line_units = []
    line_bi_flat = []

    for row in token_rows:
        phoneme = row.get('phoneme') or ''
        units = phoneme_to_units(phoneme)
        bi_pattern = row.get('bi_pattern')
        if not bi_pattern:
            bi_pattern = predict_bi_from_phoneme(phoneme)

        decoded = _decode_bi_to_syllables(units, bi_pattern)
        token_syllables = decoded['syllables_text']

        line_syllables.extend(token_syllables)
        line_units.extend(units)
        if decoded['valid']:
            line_bi_flat.extend(decoded['labels'])

        analyzed = {
            **row,
            'units': units,
            'unit_count': len(units),
            'bi_pattern': bi_pattern,
            'bi_valid': decoded['valid'],
            'bi_error': decoded['error'],
            'syllables': token_syllables,
            'syllable_count': len(token_syllables),
            'rhyme_tail_token': _extract_rhyme_tail(token_syllables[-1]) if token_syllables else '',
        }
        analyzed_tokens.append(analyzed)

    total_syllables = len(line_syllables)
    final_syllable = line_syllables[-1] if line_syllables else ''
    final_rhyme = _extract_rhyme_tail(final_syllable)

    return {
        'input_text': text,
        'tokens': analyzed_tokens,
        'line_units': line_units,
        'line_unit_count': len(line_units),
        'line_bi_pattern_flat': ''.join(line_bi_flat),
        'line_syllables': line_syllables,
        'line_syllable_count': total_syllables,
        'final_syllable': final_syllable,
        'final_rhyme': final_rhyme,
    }


def print_line_prosody_analysis(report: dict) -> None:
    print('=== ANALISIS LENGKAP LAYER-1 -> LAYER-3 ===')
    print(f"Input text         : {report['input_text']}")
    print(f"Total token        : {len(report['tokens'])}")
    print()

    print('--- Detail per token ---')
    for idx, row in enumerate(report['tokens'], start=1):
        conf = row.get('model_confidence')
        margin = row.get('model_margin')
        conf_txt = f"{conf:.3f}" if conf is not None else '-'
        margin_txt = f"{margin:.3f}" if margin is not None else '-'

        print(
            f"[{idx:02d}] token={row['token']} | raw={row['raw_label']} | final={row['label']} | "
            f"route={row['route']} | switch={row['switch_route']} | conf={conf_txt} margin={margin_txt}"
        )
        if row.get('bc_info'):
            bc = row['bc_info']
            print(f"     bc_rule={bc.get('rule_used')} | normalized={bc.get('normalized_token')}")
        if row.get('asg_info'):
            asg = row['asg_info']
            print(f"     asg_reason={asg.get('reason')} | score_en={asg.get('score_en')} | score_id={asg.get('score_id')}")
        print(f"     phoneme={row.get('phoneme', '')}")
        print(f"     units={row['units']} (n={row['unit_count']})")
        print(
            f"     bi={row.get('bi_pattern') or '-'} | bi_valid={row.get('bi_valid')}"
            + (f" | bi_error={row.get('bi_error')}" if row.get('bi_error') else '')
        )
        print(f"     syllables={row.get('syllables', [])} | syllable_count={row.get('syllable_count', 0)}")
        print(f"     token_rhyme_tail={row.get('rhyme_tail_token', '')}")

    print('\n--- Agregasi baris ---')
    print(f"line_units          : {report['line_units']}")
    print(f"line_unit_count     : {report['line_unit_count']}")
    print(f"line_bi_pattern     : {report['line_bi_pattern_flat']}")
    print(f"line_syllables      : {report['line_syllables']}")
    print(f"JUMLAH_SILABEL      : {report['line_syllable_count']}")
    print(f"SILABEL_AKHIR       : {report['final_syllable']}")
    print(f"RIMA_AKHIR          : {report['final_rhyme']}")

def _rhyme_key(rhyme_tail: str) -> str:
    t = (rhyme_tail or '').strip()
    if not t:
        return '-'
    if len(t) >= 2:
        return t[-2:]
    return t


def infer_rhyme_scheme(line_reports: list[dict]) -> str:
    keys = [_rhyme_key(rep.get('final_rhyme', '')) for rep in line_reports]
    if len(keys) == 4:
        if keys[0] == keys[1] == keys[2] == keys[3] and keys[0] != '-':
            return 'a-a-a-a'
        return 'a-b-a-b'

    mapping = {}
    next_label_ord = ord('a')
    scheme = []
    for key in keys:
        if key not in mapping:
            mapping[key] = chr(next_label_ord)
            next_label_ord += 1
        scheme.append(mapping[key])
    return '-'.join(scheme)


def analyze_pantun_multiline(text: str) -> dict:
    lines = [ln.strip() for ln in str(text).splitlines() if ln.strip()]
    line_reports = []
    for i, line in enumerate(lines, start=1):
        rep = analyze_line_prosody_full(line)
        rep['line_no'] = i
        line_reports.append(rep)

    rhyme_scheme = infer_rhyme_scheme(line_reports)
    total_syllables = sum(rep.get('line_syllable_count', 0) for rep in line_reports)

    return {
        'raw_text': text,
        'line_count': len(line_reports),
        'lines': line_reports,
        'total_syllables_all_lines': total_syllables,
        'rhyme_scheme': rhyme_scheme,
    }


def build_pantun_compact_output(report: dict) -> dict:
    lines = report.get('lines', [])

    compact = {
        'skema_rima': report.get('rhyme_scheme', ''),
        'rima_akhir_baris_1': '',
        'rima_akhir_baris_2': '',
        'rima_akhir_baris_3': '',
        'rima_akhir_baris_4': '',
        'suku_kata_baris_1': '',
        'suku_kata_baris_2': '',
        'suku_kata_baris_3': '',
        'suku_kata_baris_4': '',
    }

    for i in range(min(4, len(lines))):
        rep = lines[i]
        compact[f'rima_akhir_baris_{i+1}'] = rep.get('final_rhyme', '')
        compact[f'suku_kata_baris_{i+1}'] = rep.get('line_syllable_count', '')

    return compact


def print_pantun_multiline_analysis(report: dict, verbose: bool = False) -> None:
    compact = build_pantun_compact_output(report)
    cols = [
        'skema_rima',
        'rima_akhir_baris_1',
        'rima_akhir_baris_2',
        'rima_akhir_baris_3',
        'rima_akhir_baris_4',
        'suku_kata_baris_1',
        'suku_kata_baris_2',
        'suku_kata_baris_3',
        'suku_kata_baris_4',
    ]

    print(','.join(str(compact[col]) for col in cols))

    if not verbose:
        return

    print('\n=== ANALISIS PANTUN MULTI-BARIS (VERBOSE) ===')
    print(f"Jumlah baris terdeteksi : {report['line_count']}")
    print(f"Total silabel (semua)   : {report['total_syllables_all_lines']}")
    print(f"Skema rima              : {report['rhyme_scheme']}")

    for line_rep in report['lines']:
        print('\n' + '=' * 64)
        print(f"BARIS {line_rep['line_no']}: {line_rep['input_text']}")
        print_line_prosody_analysis(line_rep)



In [ ]:
# Contoh uji input pantun multi-baris (pisahkan dengan newline "\n")
sample_pantun = """sepulang dari surabaya
 di suruh beli kg kawat
 kecantikanmu emang tiada duanya
 yang pada glowing pun lewat"""

pantun_report = analyze_pantun_multiline(sample_pantun)

# Default: output ringkas sesuai format dataset
print_pantun_multiline_analysis(pantun_report)

# Verbose (opsional): tampilkan semua detail per token/baris
# print_pantun_multiline_analysis(pantun_report, verbose=True)
